# II-3 Docker で受信 API・MQTT・PostgreSQL・Grafana（ノートブック版）

教科書の II-3 のうち、**Pi と Docker がなくても試せる部分**を1冊にまとめたものです。Docker の代わりに、このノートブックの中で PostgreSQL のサーバーを1つ立て（Colab では apt で入れ、Mac ではすでに入っているものを使う）、本文の `01_schema.sql`・`api.py`・`mqtt2pg.py` の受け取りの部分・Grafana の SQL・`sqlite2pg.py`・バックアップと復元・`bench_analysis.py` を、上から順に試します（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約10秒。Colab では PostgreSQL を入れるのに1分ほどかかります）。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | PostgreSQL を 127.0.0.1 だけで立て、`01_schema.sql` で表を作る | 手順 8〜9（compose の db） | `db/init/01_schema.sql` |
| 2 | 受信 API：`/health`・`/ingest`・`/latest`、項目の足りない JSON、DB が止まったとき | 手順 9 の 4) | `app/api.py` |
| 3 | MQTT の記録係の受け取りの部分（JSON・JSON でない文字・数だけ） | 手順 9 の 4) | `app/mqtt2pg.py` の `on_message` |
| 4 | Grafana のパネルの SQL（JSONB から温度を取り出す） | 手順 11 | パネルの SQL の例 |
| 5 | 第0章の SQLite の記録を移す：`--dry-run` → 本番 → 二重に移さない | 手順 13 | `sqlite2pg.py` |
| 6 | バックアップ（`pg_dump -Fc`）を別の DB に復元して、件数が同じか確かめる | 手順 14 | — |
| 7 | 新旧の構成の遅れの比較：順番を乱数で混ぜた計画と、まとめて測る誤った計画 | 手順 10 | `bench_analysis.py`、`synth_bench.py`（ノートブック版で追加） |

- 次のセルたちが、II-3-6 のファイルを、このノートブックの隣に `%%writefile` で書き出します。中身は II-3-6 と同じです。変えたのは、`bench_analysis.py` の図の日本語フォントの候補を足したこと（`plt.rcParams["font.family"]` の行）と、Grafana の SQL の `$__timeFilter(recv_time)`（Grafana が画面で選んだ時間の範囲に置き換えるマクロ）を、ふつうの SQL の範囲に書きかえたことだけです。
- **変えた使い方**：本文の DB はコンテナの中の PostgreSQL（`127.0.0.1:5432`、ユーザーと DB は `lab`）です。ここでは、ほかの PostgreSQL と重ならないように **5544 番**で立て、環境変数 `PGPORT=5544` で知らせます（`api.py`・`mqtt2pg.py`・`sqlite2pg.py` は libpq の環境変数 `PGHOST`・`PGPORT`・`PGUSER`・`PGPASSWORD`・`PGDATABASE` を読むので、コードは変えません）。パスワードは、本文の `.env` の代わりに、ノートブックが実行のたびに乱数で作ります（ファイルにもノートブックにも残しません）。PostgreSQL の版は、Colab の apt のもの・Mac に入っているものになり、本文の `postgres:17` とは違うことがあります（この章の SQL の範囲では違いは出ないはずです）。
- `api.py` は、本文では uvicorn で 127.0.0.1:8001 に立てます。ここでは FastAPI の `TestClient` で、ポートを開かずに同じ要求を送ります。`mqtt2pg.py` はブローカ（Mosquitto）につながず、受け取ったときに呼ばれる `on_message` に、作ったメッセージを直接渡します。
- 7 節の `bench.csv` は、`synth_bench.py`（本文にはない）が作る合成データで、**実測の値ではありません**。`bench_e2e.py`（Pi の上で MQTT で送り、SQLite と PostgreSQL から読む）は、ブローカが2つ要るので入れていません。
- **入れていないもの（Pi・Docker が要る部分）**：Docker Engine の導入、`compose.yaml` での起動、Docker が ufw を素通りすることの確かめ、Mosquitto のコンテナ、Grafana の画面、`docker stats` でのメモリ、`bench_e2e.py` での実測、移行（ポートの切り替え）。`compose.yaml` は下に読むためだけに載せてあります。
- ⚠ 本文と同じく、パスワードをコマンド行やノートブックに書かないでください。ここでは環境変数で渡しています。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab に fastapi・psycopg・paho-mqtt が入っているとは限らない。版は要確認）
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ.setdefault(_v, "1")   # 小さい行列の計算だけなので1スレッドにする（CPU が混んでいると、多スレッドはかえって遅くなる）
import importlib, subprocess, sys
for mod, pkg in [("psycopg", "psycopg[binary]"), ("fastapi", "fastapi"), ("httpx", "httpx"), ("paho.mqtt", "paho-mqtt"),
                 ("statsmodels", "statsmodels"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import psycopg, fastapi, pydantic, paho.mqtt, statsmodels, pandas
print("OK", "psycopg", psycopg.__version__, "fastapi", fastapi.__version__, "pydantic", pydantic.__version__,
      "paho-mqtt", paho.mqtt.__version__, "statsmodels", statsmodels.__version__, "pandas", pandas.__version__)
print("（ノートブックは psycopg 3.3.9・fastapi 0.165.0・pydantic 2.14.4・paho-mqtt 2.2.0・statsmodels 0.15.0 で確かめた）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

**Mac｜~/home-lab/partII/ii3/stack/compose.yaml**（読むだけ。本文の構成の全体）

```yaml
# II-3：受信API ＋ Mosquitto ＋ PostgreSQL ＋ Grafana（Raspberry Pi 5・4GB 向けの軽い構成）
# 置き場所：Mac の ~/home-lab/partII/ii3/stack/（Git で管理）→ Pi の ~/stack/ に rsync で送る
# 秘密の値は .env に書く（.env は Git に入れない。.env.example を写して作る）
#
# ネットワークの方針（Docker が ufw を素通りする問題への対策）
#   ・LAN から届く必要があるもの（MQTT）は network_mode: host にする → ホストの ufw がそのまま効く
#   ・DB と Grafana は ports を「127.0.0.1:」付きで公開 → Pi の外からは届かない（見るときは SSH トンネル）
#   ・自作のプログラム（api・mqtt2pg）も host にして、127.0.0.1 で DB と Mosquitto に話す
services:
  mosquitto:
    image: eclipse-mosquitto:2.0.20          # 版は Docker Hub で要確認。決めたら固定する
    network_mode: host
    restart: unless-stopped
    volumes:
      - ./mosquitto/config:/mosquitto/config:ro   # 読み取り専用（:ro がないと、起動時に持ち主が uid 1883 に書き換えられる）
      - mosquitto-data:/mosquitto/data
    mem_limit: 64m                           # 上限が効いているかは docker stats で確かめる（II-3-5 手順9）

  db:
    image: postgres:17                       # arm64 版がある。版は公式で要確認
    restart: unless-stopped
    environment:
      POSTGRES_USER: lab
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD:?.env に POSTGRES_PASSWORD を書く}
      POSTGRES_DB: lab
    command: ["postgres", "-c", "shared_buffers=64MB", "-c", "max_connections=20"]
    ports:
      - "127.0.0.1:5432:5432"                # 127.0.0.1 だけ。これを "5432:5432" にすると LAN に丸見えになる
    volumes:
      - db-data:/var/lib/postgresql/data
      - ./db/init:/docker-entrypoint-initdb.d:ro    # 初回だけ実行される（表を作る）
    mem_limit: 384m
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U lab -d lab"]
      interval: 10s
      timeout: 5s
      retries: 5

  grafana:
    image: grafana/grafana-oss:11.6.0        # 版は公式で要確認
    restart: unless-stopped
    environment:
      GF_SECURITY_ADMIN_USER: admin
      GF_SECURITY_ADMIN_PASSWORD: ${GRAFANA_ADMIN_PASSWORD:?.env に GRAFANA_ADMIN_PASSWORD を書く}
      GF_USERS_ALLOW_SIGN_UP: "false"
      GF_ANALYTICS_REPORTING_ENABLED: "false"
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD}
    ports:
      - "127.0.0.1:3000:3000"
    volumes:
      - grafana-data:/var/lib/grafana
      - ./grafana/provisioning:/etc/grafana/provisioning:ro
    depends_on:
      db:
        condition: service_healthy
    mem_limit: 256m

  api:
    build: ./app
    image: home-lab-app:0.1
    network_mode: host
    restart: unless-stopped
    command: ["uvicorn", "api:app", "--host", "127.0.0.1", "--port", "8001"]
    environment:
      PGHOST: 127.0.0.1
      PGUSER: lab
      PGPASSWORD: ${POSTGRES_PASSWORD}
      PGDATABASE: lab
    depends_on:
      db:
        condition: service_healthy
    mem_limit: 128m

  mqtt2pg:
    image: home-lab-app:0.1
    network_mode: host
    restart: unless-stopped
    command: ["python", "mqtt2pg.py"]
    environment:
      MQTT_HOST: 127.0.0.1
      MQTT_PORT: ${MQTT_PORT:-1884}
      MQTT_USER: logger
      MQTT_PASS: ${MQTT_LOGGER_PASSWORD:?.env に MQTT_LOGGER_PASSWORD を書く}
      PGHOST: 127.0.0.1
      PGUSER: lab
      PGPASSWORD: ${POSTGRES_PASSWORD}
      PGDATABASE: lab
    depends_on:
      db:
        condition: service_healthy
      api:
        condition: service_started
    mem_limit: 96m

volumes:
  mosquitto-data:
  db-data:
  grafana-data:
```

次のセルたちで、表の定義・受信 API・MQTT の記録係・移行のプログラム・遅れの解析と、合成データのプログラムを書き出します。

In [ ]:
%%writefile 01_schema.sql
-- 初回の起動のときだけ実行される（db-data のボリュームが空のとき）
CREATE TABLE IF NOT EXISTS messages (
    id         BIGSERIAL PRIMARY KEY,
    recv_unix  DOUBLE PRECISION NOT NULL,          -- 受け取った時刻（UNIX 秒。第0章の SQLite と同じ）
    recv_time  TIMESTAMPTZ NOT NULL DEFAULT now(),
    topic      TEXT NOT NULL,
    dev        TEXT,
    seq        INTEGER,
    payload    JSONB NOT NULL                      -- 受け取った JSON をそのまま入れる
);
CREATE INDEX IF NOT EXISTS idx_messages_dev_time ON messages (dev, recv_time);

In [ ]:
%%writefile api.py
"""II-3：データ受信 API（FastAPI）。HTTP の POST で JSON を受け取り、PostgreSQL に入れる

  POST /ingest   … {"dev": "pico1", "seq": 12, ...} を受け取って保存する（トピックは http/<dev>）
  GET  /health   … DB につながるか
  GET  /latest   … ?dev=pico1&n=5 で最新の n 件
II-4 で Pico から HTTP で送る比較に使う。II-3 では 127.0.0.1:8001 にだけ公開する。
"""
import json
import time

import psycopg
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict

app = FastAPI(title="home-lab ingest")


class Msg(BaseModel):
    model_config = ConfigDict(extra="allow")     # dev・seq 以外の項目もそのまま受け取る
    dev: str
    seq: int | None = None


def conn():
    return psycopg.connect(connect_timeout=3)    # 接続先は環境変数 PGHOST・PGUSER など


@app.get("/health")
def health():
    try:
        with conn() as c:
            c.execute("SELECT 1")
        return {"ok": True}
    except Exception as e:
        raise HTTPException(status_code=503, detail=str(e))


@app.post("/ingest")
def ingest(m: Msg):
    t = time.time()
    body = m.model_dump()
    with conn() as c:
        c.execute("INSERT INTO messages (recv_unix, topic, dev, seq, payload) VALUES (%s, %s, %s, %s, %s)",
                  (t, f"http/{m.dev}", m.dev, m.seq, json.dumps(body)))
    return {"ok": True, "recv_unix": t}


@app.get("/latest")
def latest(dev: str, n: int = 5):
    with conn() as c:
        rows = c.execute("SELECT recv_time, seq, payload FROM messages WHERE dev = %s ORDER BY id DESC LIMIT %s",
                         (dev, min(n, 100))).fetchall()
    return [{"recv_time": r[0].isoformat(), "seq": r[1], "payload": r[2]} for r in rows]

In [ ]:
%%writefile mqtt2pg.py
"""II-3：MQTT（lab/#）を購読して PostgreSQL に保存する（第0章の pi_mqtt_logger.py の PostgreSQL 版）"""
import json
import os
import time

import paho.mqtt.client as mqtt
import psycopg

con = psycopg.connect(autocommit=True)           # 接続先は環境変数 PGHOST・PGUSER など


def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code.is_failure:
        print("接続失敗:", reason_code, flush=True)
        return
    client.subscribe("lab/#", qos=0)
    print("購読: lab/#", flush=True)


def on_message(client, userdata, msg):
    t = time.time()
    text = msg.payload.decode("utf-8", errors="replace")
    try:
        d = json.loads(text)
        if not isinstance(d, dict):
            d = {"value": d}
    except json.JSONDecodeError:
        d = {"raw": text}
    con.execute("INSERT INTO messages (recv_unix, topic, dev, seq, payload) VALUES (%s, %s, %s, %s, %s)",
                (t, msg.topic, d.get("dev"), d.get("seq"), json.dumps(d)))


def main():
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="pg-logger")
    client.username_pw_set(os.environ["MQTT_USER"], os.environ["MQTT_PASS"])
    client.on_connect = on_connect
    client.on_message = on_message
    client.reconnect_delay_set(min_delay=1, max_delay=30)
    client.connect(os.environ.get("MQTT_HOST", "127.0.0.1"), int(os.environ.get("MQTT_PORT", "1883")), keepalive=60)
    client.loop_forever()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile sqlite2pg.py
#!/usr/bin/env python3
"""II-3：第0章の SQLite（~/lab/lab.db）の記録を、PostgreSQL の messages 表に移す（Raspberry Pi の上で実行）

使い方（Pi の ~/stack で）:
  set -a; . ./.env; set +a                     # .env の値を環境変数に読み込む（パスワードをコマンド行に書かない）
  export PGPASSWORD="$POSTGRES_PASSWORD"
  ~/labenv/bin/python sqlite2pg.py ~/lab/lab.db --dry-run
  （件数を確かめてから --dry-run を外して実行する）
二重に移さないように、SQLite の最初と最後の記録と「受け取った時刻（recv_unix）と topic がまったく同じ行」が
PostgreSQL にすでにあれば止まる（--force で無視）。手順7（dev = test）や手順10（dev = bench）で
新しい構成に入った行は、受け取った時刻が違うので、この確かめ方では引っかからない。
"""
import argparse
import json
import sqlite3
import sys

import psycopg


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("sqlite")
    ap.add_argument("--dry-run", action="store_true")
    ap.add_argument("--force", action="store_true")
    a = ap.parse_args()
    src = sqlite3.connect(a.sqlite)
    rows = src.execute("SELECT recv_unix, topic, payload, dev, seq FROM messages "
                       "WHERE dev IS NULL OR dev != 'bench' ORDER BY id").fetchall()
    if not rows:
        sys.exit("SQLite に移す記録がありません")
    t0, t1 = rows[0][0], rows[-1][0]
    print(f"SQLite：{len(rows)} 件（{t0:.0f}〜{t1:.0f}、UNIX 秒）")
    with psycopg.connect(host="127.0.0.1", user="lab", dbname="lab") as pg:
        n = 0
        for recv, topic, *_ in (rows[0], rows[-1]):
            n += pg.execute("SELECT count(*) FROM messages WHERE recv_unix = %s AND topic = %s",
                            (recv, topic)).fetchone()[0]
        print(f"PostgreSQL にある、SQLite の最初と最後の記録と同じ行：{n} 件")
        if n and not a.force:
            sys.exit("すでに移した可能性があります（二重に入れないように止めました。分かったうえで --force）")
        if a.dry_run:
            print("--dry-run なので書き込みません")
            return
        out = []
        for recv, topic, payload, dev, seq in rows:
            try:
                d = json.loads(payload)
                d = d if isinstance(d, dict) else {"value": d}
            except json.JSONDecodeError:
                d = {"raw": payload}
            out.append((recv, topic, dev, seq, json.dumps(d)))
        with pg.cursor() as cur:
            cur.executemany("INSERT INTO messages (recv_unix, recv_time, topic, dev, seq, payload) "
                            "VALUES (%s, to_timestamp(%s), %s, %s, %s, %s)",
                            [(r[0], r[0], r[1], r[2], r[3], r[4]) for r in out])
        print(f"{len(out)} 件を移しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile bench_analysis.py
#!/usr/bin/env python3
"""II-3：bench_e2e.py の結果（bench.csv）で、旧構成（SQLite）と新構成（Docker＋PostgreSQL）の遅れを比べる

使い方: cd ~/home-lab/partII/ii3 && uv run python bench_analysis.py bench.csv

やること
  1. 構成ごとの欠損率（Wilson 区間）と、遅れの中央値・95%点・99%点
  2. ブロックごとの中央値を「測った順番」に並べて描き、時間とともに変わる傾向（ドリフト）がないか見る
  3. ブロックの中央値を応答にして、遅れ ~ 構成 ＋ 実行（何回目か）＋ 順番 を回帰する
     実行ごとの全体の高さの違いは「実行」の項が受け持ち、順番の係数は「1回の実行の中での変化」になる
     順番を乱数で混ぜてあるので、ドリフトがあっても構成の差はかたよらない（ランダム化の効果）
     【誤りの例】もし片方の構成を先にまとめて測っていたら、ドリフトと構成の差は区別できない
"""
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False


def main(path):
    d = pd.read_csv(path)
    print(f"{d['run'].nunique()} 回の実行、{len(d)} 通")
    print(f"{'構成':8s} {'受信':>11s} {'欠損率[%]':>9s} {'(Wilson)':>14s} {'中央値':>8s} {'95%点':>8s} {'99%点':>8s}  [ms]")
    for st, g in d.groupby("stack"):
        x = g["latency_ms"].dropna()
        k, n = int(g["latency_ms"].isna().sum()), len(g)
        lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
        print(f"{st:8s} {n-k:5d}/{n:<5d} {k/n*100:9.2f} {f'({lo*100:.2f}〜{hi*100:.2f})':>14s} "
              f"{x.median():8.2f} {x.quantile(0.95):8.2f} {x.quantile(0.99):8.2f}")

    b = (d.dropna(subset=["latency_ms"]).groupby(["run", "stack", "order"])["latency_ms"]
         .median().rename("med").reset_index())
    runs = {r: i for i, r in enumerate(d["run"].unique())}        # 実行した順（CSV に追記された順）
    # 実行ごとに高さが違ってよい（C(run)）。順番 order は実行の中での位置なので、c は「実行の中での変化」
    # 【誤りの例】順番を order + 1000·(何回目か) のように通し番号にすると、実行どうしの高さの違いを
    #   1000 ブロック分の傾きと取り違え、c がほぼ 0 になってしまう
    m = smf.ols("med ~ C(stack, Treatment('sqlite')) + C(run) + order", b).fit()
    name = [p for p in m.params.index if p.startswith("C(stack")][0]
    est, (lo, hi) = m.params[name], m.conf_int().loc[name]
    slope = m.params["order"]
    print(f"\nブロックの中央値（{len(b)} ブロック、{len(runs)} 回の実行）の回帰：遅れ = a + b·(docker か) + (実行ごとの高さ) + c·順番")
    print(f"  docker − sqlite = {est:+.2f} ms（95%区間 {lo:+.2f}〜{hi:+.2f}）")
    print(f"  順番が1つ進むごとの変化 c = {slope:+.3f} ms（{'時間とともに変わる傾向がある' if m.pvalues['order'] < 0.05 else 'はっきりした傾向は見えない'}）")
    print("  独立な単位は「ブロック」。1通ずつを独立とみなすと、ブロックの中の似た値を何百回も数えることになる。")
    naive = b.groupby("stack")["med"].mean()
    print(f"  （参考）ブロックの平均の単純な差：{naive.get('docker', np.nan) - naive.get('sqlite', np.nan):+.2f} ms")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
    for st, g in d.groupby("stack"):
        x = np.sort(g["latency_ms"].dropna().to_numpy())
        ax[0].step(x, np.arange(1, len(x) + 1) / len(x), where="post", label=st)
    b["x"] = b["order"] + (b["order"].max() + 3) * b["run"].map(runs)   # 図だけ：実行ごとに右へずらして並べる
    for st, g in b.groupby("stack"):
        ax[1].plot(g["x"], g["med"], "o", label=st)
    ax[0].set(xscale="log", xlabel="遅れ [ms]（対数目盛）", ylabel="累積割合", title="遅れの経験分布関数")
    ax[1].set(xlabel="測った順番（実行ごとに右へずらしてある）", ylabel="ブロックの中央値 [ms]", title="順番に並べたブロックの中央値（ドリフトの確認）")
    for x in ax:
        x.grid(True, which="both", alpha=0.3)
        x.legend()
    fig.tight_layout()
    fig.savefig("bench.png", dpi=130)
    print("\nbench.png を保存しました")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        sys.exit("使い方: uv run python bench_analysis.py bench.csv")
    main(sys.argv[1])

In [ ]:
%%writefile grafana_panel.sql
-- Grafana のパネル（データソース lab-postgres、Code モード）に書く SQL の例
-- $__timeFilter は Grafana が「画面で選んだ時間の範囲」に置き換えるマクロ（Grafana のドキュメントで要確認）
SELECT
  recv_time AS "time",
  (payload->>'temp_c')::float AS "Pico 内蔵温度 [°C]"
FROM messages
WHERE dev = 'pico1' AND $__timeFilter(recv_time)
ORDER BY 1;

In [ ]:
%%writefile synth_bench.py
#!/usr/bin/env python3
"""II-3 のノートブック版で追加：bench_e2e.py の bench.csv と同じ形の、合成の遅れのデータを作る

bench_e2e.py と同じく、構成ごとに --blocks ブロック（1ブロック --n 通）を作り、ブロックの順番を乱数で混ぜる
（--sequential を付けると、旧構成をまとめて先に・新構成をあとに測る「誤った計画」になる）。
遅れ [ms] = 構成の基準（sqlite 2.6、docker 2.6 + DIFF）＋ 実行ごとの高さ ＋ ドリフト（順番とともに上がって頭打ち）
          ＋ ブロックごとのゆらぎ ＋ 1通ごとの裾の重いゆらぎ（まれに数十 ms）。ごくまれに届かない（空欄）
使い方: python synth_bench.py --out bench_synth.csv [--runs 2] [--blocks 10] [--n 100] [--seed 1] [--sequential]
"""
import argparse
import csv
import json
import random
import uuid

import numpy as np

BASE, DIFF = 2.6, 0.8                 # 旧構成の基準 [ms]、新構成 − 旧構成の真の差 [ms]
DRIFT_MAX, DRIFT_TAU = 0.8, 6.0       # ドリフト：DRIFT_MAX·(1 − exp(−順番/DRIFT_TAU)) [ms]（温まる・たまるなどのまね）
SD_RUN, SD_BLOCK = 0.3, 0.12          # 実行ごと・ブロックごとのゆらぎ [ms]
LOSS = 0.001


def drift(order):
    return DRIFT_MAX * (1 - np.exp(-order / DRIFT_TAU))


def make_plan(blocks, seed, sequential):
    plan = ["sqlite"] * blocks + ["docker"] * blocks
    if not sequential:
        random.Random(seed).shuffle(plan)           # bench_e2e.py と同じ混ぜ方
    return plan


def one_run(rng, plan, n):
    run = uuid.UUID(int=int(rng.integers(0, 2**63))).hex[-8:]
    h = rng.normal(0, SD_RUN)
    rows = []
    for order, stack in enumerate(plan):
        mu = BASE + (DIFF if stack == "docker" else 0.0) + h + drift(order) + rng.normal(0, SD_BLOCK)
        x = mu * rng.lognormal(0, 0.15, n)
        spike = rng.random(n) < 0.01
        x[spike] += rng.exponential(20, spike.sum())   # まれに数十 ms（ほかの処理の割り込みなど）
        for k in range(n):
            lost = rng.random() < LOSS
            rows.append([run, stack, order, k, "" if lost else f"{x[k]:.3f}"])
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="bench_synth.csv")
    ap.add_argument("--runs", type=int, default=2)
    ap.add_argument("--blocks", type=int, default=10)
    ap.add_argument("--n", type=int, default=100)
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--sequential", action="store_true", help="旧構成を先にまとめて測る（誤った計画の例）")
    a = ap.parse_args()
    rng = np.random.default_rng(a.seed)
    rows = []
    for r in range(a.runs):
        rows += one_run(rng, make_plan(a.blocks, a.seed * 100 + r, a.sequential), a.n)
    with open(a.out, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["run", "stack", "order", "seq", "latency_ms"])
        w.writerows(rows)
    json.dump({"docker_minus_sqlite_ms": DIFF}, open(a.out.replace(".csv", "_truth.json"), "w"))
    print(f"{a.out}：{a.runs} 回の実行 × {2 * a.blocks} ブロック × {a.n} 通（{'まとめて順に' if a.sequential else '順番は乱数'}）。"
          f"真の差 docker − sqlite = {DIFF} ms")


if __name__ == "__main__":
    main()

In [ ]:
import glob, json, os, re, secrets, shlex, shutil, sqlite3, time
import numpy as np
import pandas as pd
from IPython.display import Image, display


def run(args, check=True):
    """ターミナルの「~/labenv/bin/python ○○.py …」の代わり。表示を出し、(終了コード, 表示) を返す"""
    p = subprocess.run([sys.executable] + shlex.split(args), capture_output=True, text=True)
    print(p.stdout, end="")
    keep = [l for l in p.stderr.splitlines() if "findfont" not in l and "font cache" not in l and l.strip()]
    if keep:
        print("\n".join(keep), file=sys.stderr)
    if check:
        assert p.returncode == 0, "★実行に失敗：" + args
    return p.returncode, p.stdout + p.stderr


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1):
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))


print("準備ができました")

## 1. PostgreSQL を 127.0.0.1 だけで立てる（compose の db の代わり）

本文の `compose.yaml` の db は、`postgres:17` のイメージで、`POSTGRES_USER: lab`・`POSTGRES_DB: lab`、ポートは `127.0.0.1:5432` にだけ出し、最初の起動のときに `db/init/01_schema.sql` を実行します。ここでは同じことを、PostgreSQL のコマンド（`initdb`・`pg_ctl`）で行います。

- 待ち受けは `listen_addresses=127.0.0.1` で、LAN 側からは届きません（本文の「DB は 127.0.0.1 にだけ出す」と同じ）。ファイルのソケットは使いません（`unix_socket_directories=''`）。
- `shared_buffers=64MB` は本文の compose と同じ小さめの設定です。
- Colab（root で動いている）では、PostgreSQL が root での起動を断るので、`pgnb` というユーザーを作って、そのユーザーで動かします。

In [ ]:
import socket
PGPORT = 5544                              # 本文は 5432（コンテナ）。ほかの PostgreSQL と重ならない番号にする
def _used(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0
while _used(PGPORT):     # 使われていたら（前の実行の残りなど）次の番号へ
    PGPORT += 1
ROOT = hasattr(os, "geteuid") and os.geteuid() == 0


def find_bindir():
    cands = [os.path.dirname(shutil.which("pg_ctl") or "")]
    for pat in ["/usr/lib/postgresql/*/bin", "/Library/PostgreSQL/*/bin", "/opt/homebrew/opt/postgresql*/bin", "/usr/local/opt/postgresql*/bin"]:
        cands += sorted(glob.glob(pat), reverse=True)
    for c in cands:
        if c and os.path.exists(os.path.join(c, "initdb")):
            return c
    return None


PGBIN = find_bindir()
if PGBIN is None and shutil.which("apt-get"):              # Colab：PostgreSQL を入れる（1分ほど）
    subprocess.run("apt-get -qq update > /dev/null && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y postgresql > /dev/null",
                   shell=True, check=True)
    PGBIN = find_bindir()
assert PGBIN, "PostgreSQL のコマンドが見つかりません（Mac なら公式のインストーラか Homebrew で入れる）"
print("PostgreSQL のコマンド:", PGBIN)
print(subprocess.run([f"{PGBIN}/postgres", "--version"], capture_output=True, text=True).stdout)


def as_pg(cmd):
    """root のときは pgnb ユーザーで実行する"""
    return ["runuser", "-u", "pgnb", "--"] + cmd if ROOT else cmd


W = os.path.abspath("pgwork")
DATA = os.path.join(W, "data")
if os.path.exists(os.path.join(DATA, "postmaster.pid")):   # もう一度実行したとき：前のサーバーを止める
    subprocess.run(as_pg([f"{PGBIN}/pg_ctl", "-D", DATA, "-w", "-m", "fast", "stop"]), capture_output=True)
shutil.rmtree(W, ignore_errors=True)
os.makedirs(W)
if ROOT:
    subprocess.run("id pgnb > /dev/null 2>&1 || useradd -m pgnb", shell=True, check=True)
    shutil.chown(W, "pgnb", "pgnb")
PGPASSWORD = secrets.token_urlsafe(18)                   # 本文の .env の POSTGRES_PASSWORD の代わり（実行のたびに作る）
pwfile = os.path.join(W, "pw.txt")
with open(pwfile, "w") as f:
    f.write(PGPASSWORD + "\n")
os.chmod(pwfile, 0o600)
if ROOT:
    shutil.chown(pwfile, "pgnb", "pgnb")
r = subprocess.run(as_pg([f"{PGBIN}/initdb", "-D", DATA, "-U", "lab", "-A", "scram-sha-256", "--pwfile", pwfile,
                          "-E", "UTF8", "--locale=C"]), capture_output=True, text=True)
os.remove(pwfile)                                        # パスワードのファイルはすぐ消す
assert r.returncode == 0, r.stdout + r.stderr
opts = f"-p {PGPORT} -c listen_addresses=127.0.0.1 -c unix_socket_directories='' -c shared_buffers=64MB"
r = subprocess.run(as_pg([f"{PGBIN}/pg_ctl", "-D", DATA, "-l", os.path.join(DATA, "server.log"), "-w", "-o", opts, "start"]),
                   capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0
# 本文の compose で app に渡している環境変数と同じもの（番号だけ PGPORT）
os.environ.update(PGHOST="127.0.0.1", PGPORT=str(PGPORT), PGUSER="lab", PGPASSWORD=PGPASSWORD, PGDATABASE="lab")
with psycopg.connect(dbname="postgres", autocommit=True) as c:      # POSTGRES_DB: lab の代わり
    c.execute("CREATE DATABASE lab")
with psycopg.connect(autocommit=True) as c:                          # db/init/01_schema.sql の代わり
    c.execute(open("01_schema.sql").read())
    cols = c.execute("SELECT column_name, data_type FROM information_schema.columns "
                     "WHERE table_name = 'messages' ORDER BY ordinal_position").fetchall()
    idx = [r[0] for r in c.execute("SELECT indexname FROM pg_indexes WHERE tablename = 'messages'")]
    listen = c.execute("SHOW listen_addresses").fetchone()[0]
for col in cols:
    print(col)
ok([c for c, _ in cols] == ["id", "recv_unix", "recv_time", "topic", "dev", "seq", "payload"] and dict(cols)["payload"] == "jsonb",
   "messages 表ができた（payload は JSONB）")
ok("idx_messages_dev_time" in idx, "(dev, recv_time) の索引ができた")
ok(listen == "127.0.0.1", "127.0.0.1 だけで待ち受けている")

## 2. 受信 API（`api.py`）

本文では、Pi の上で `curl -s http://127.0.0.1:8001/health` を実行し、II-4 では Pico から `POST /ingest` で送ります。ここでは FastAPI の `TestClient` で、同じ要求を送ります（ポートは開きません）。

In [ ]:
from fastapi.testclient import TestClient
import api
cl = TestClient(api.app)
r = cl.get("/health"); print("GET /health", r.status_code, r.json())
ok(r.status_code == 200 and r.json() == {"ok": True}, "/health：DB につながる")
for k in range(3):
    r = cl.post("/ingest", json={"dev": "pico2", "seq": k, "temp_c": 24.5 + k, "rssi": -60})
    print("POST /ingest", r.status_code, r.json())
ok(r.status_code == 200 and r.json()["ok"], "/ingest：受け取って保存した（受け取った時刻 recv_unix を返す）")
r = cl.get("/latest", params={"dev": "pico2", "n": 2}); print("GET /latest", json.dumps(r.json(), ensure_ascii=False, indent=1))
lat = r.json()
ok([x["seq"] for x in lat] == [2, 1], "/latest?dev=pico2&n=2：新しい順に 2 件")
ok(lat[0]["payload"] == {"dev": "pico2", "seq": 2, "temp_c": 26.5, "rssi": -60},
   "dev・seq 以外の項目（temp_c・rssi）も、そのまま JSONB に入っている（extra=\"allow\"）")
with psycopg.connect() as c:
    topic = c.execute("SELECT DISTINCT topic FROM messages WHERE dev = 'pico2'").fetchall()
ok(topic == [("http/pico2",)], "トピックは http/<dev>（MQTT の lab/… と区別できる）")

In [ ]:
# 項目が足りない・型が違う JSON は、DB に触れる前に断られる（pydantic の検査）
r1 = cl.post("/ingest", json={"seq": 1, "temp_c": 20.0})
r2 = cl.post("/ingest", json={"dev": "pico2", "seq": "abc"})
print(r1.status_code, r1.json()["detail"][0]["msg"])
print(r2.status_code, r2.json()["detail"][0]["msg"])
ok(r1.status_code == 422 and r2.status_code == 422, "dev がない・seq が数でない JSON は 422 で断られる")
# DB が止まっている（つながらない）とき：/health は 503（compose の healthcheck などで気づける）
os.environ["PGPORT"] = str(PGPORT + 100)          # わざと違う（だれも待ち受けていない）番号にする
r = cl.get("/health")
os.environ["PGPORT"] = str(PGPORT)               # 元に戻す
print(r.status_code, r.json()["detail"][:80])
ok(r.status_code == 503, "DB につながらないとき、/health は 503")

## 3. MQTT の記録係（`mqtt2pg.py`）の受け取りの部分

`mqtt2pg.py` は、読み込んだときに `psycopg.connect(autocommit=True)` で DB につなぎ（接続先は環境変数）、`main()` でブローカにつないで `lab/#` を購読します。ここでは `main()` は呼ばず、メッセージが届いたときに paho-mqtt が呼ぶ `on_message(client, userdata, msg)` に、`topic` と `payload`（バイト列）を持った作りもののメッセージを渡します。

In [ ]:
import mqtt2pg


class FakeMsg:
    """paho-mqtt の MQTTMessage の代わり（on_message が使う topic と payload だけ）"""
    def __init__(self, topic, payload):
        self.topic, self.payload = topic, payload


tests = [("lab/pico1/telemetry", json.dumps({"dev": "pico1", "seq": 7, "temp_c": 23.9}).encode()),
         ("lab/test/telemetry", b'{"dev":"test","seq":1}'),          # 本文の手順 9 の mosquitto_pub と同じ中身
         ("lab/raw", "こんにちは".encode()),                            # JSON でない
         ("lab/num", b"42"),                                          # JSON だが dict でない
         ("lab/bad", b"\xff\xfe{")]                                    # UTF-8 でもない
for t, p in tests:
    mqtt2pg.on_message(None, None, FakeMsg(t, p))
with psycopg.connect() as c:
    got = c.execute("SELECT topic, dev, seq, payload FROM messages WHERE topic LIKE 'lab/%' ORDER BY id").fetchall()
for g in got:
    print(g)
ok(got[0] == ("lab/pico1/telemetry", "pico1", 7, {"dev": "pico1", "seq": 7, "temp_c": 23.9}), "JSON の dev・seq は列にも入る")
ok(got[2][1:] == (None, None, {"raw": "こんにちは"}), "JSON でない文字は {\"raw\": …} として保存される（捨てない）")
ok(got[3][3] == {"value": 42}, "dict でない JSON は {\"value\": …}")
ok(got[4][3]["raw"].startswith("\ufffd"), "UTF-8 でないバイト列も、置き換え文字（U+FFFD）で読んで保存し、落ちない")

## 4. Grafana のパネルの SQL（手順 11）

本文の SQL の例は、Grafana の `$__timeFilter(recv_time)` マクロ（画面で選んだ時間の範囲に置き換わる）を使っています。ここでは、その部分を「直近 1 時間」のふつうの SQL に書きかえて実行します。`payload->>'temp_c'` は JSONB から文字として取り出し、`::float` で数にします。まず、`pico1` の記録を 5 秒おきに 20 件入れておきます。

In [ ]:
t_now = time.time()
with psycopg.connect() as c:
    for k in range(20):
        t = t_now - (19 - k) * 5
        c.execute("INSERT INTO messages (recv_unix, recv_time, topic, dev, seq, payload) VALUES (%s, to_timestamp(%s), %s, %s, %s, %s)",
                  (t, t, "lab/pico1/telemetry", "pico1", 100 + k, json.dumps({"dev": "pico1", "seq": 100 + k, "temp_c": 24.0 + 0.05 * k})))
GRAFANA_SQL = open("grafana_panel.sql").read()
sql = GRAFANA_SQL.replace("$__timeFilter(recv_time)", "recv_time BETWEEN now() - interval '1 hour' AND now()")   # ノートブック版：マクロを書きかえる
print(sql)
with psycopg.connect() as c:
    rows = c.execute(sql).fetchall()
print(len(rows), "行", rows[:2], "…", rows[-1:])
ok(len(rows) == 21 and all(isinstance(v, float) for _, v in rows),
   "直近 1 時間の pico1 の温度が 21 行（いま入れた 20 行と、3 節の mqtt2pg の 1 行）、数（float）として取り出せる")
ok(all(rows[i][0] <= rows[i + 1][0] for i in range(len(rows) - 1)), "時刻の順に並ぶ（ORDER BY 1）")

## 5. 第0章の SQLite の記録を移す（手順 13、`sqlite2pg.py`）

第0章の記録係（`pi_mqtt_logger.py`）が作る `~/lab/lab.db` の代わりに、同じ表の形（第0章の `init_db` の `CREATE TABLE`）の `lab/lab.db` を作り、次の行を入れます。

- `pico1` の記録 720 件（5 秒おき、1 時間分。執筆時より前の日時）
- 手順 10 の遅れの比較で入った `dev = bench` の記録 50 件（移さない）
- JSON でない記録 1 件と、dict でない JSON 1 件（第0章の記録係は、受け取った文字をそのまま保存していた）

そのあと、本文と同じ順に、`--dry-run` で件数を確かめ → `--dry-run` を外して移し → **もう一度実行してしまった**ときに止まることを確かめます。

In [ ]:
os.makedirs("lab", exist_ok=True)
if os.path.exists("lab/lab.db"):
    os.remove("lab/lab.db")
con = sqlite3.connect("lab/lab.db")
con.execute("""CREATE TABLE IF NOT EXISTS messages(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    recv_unix REAL NOT NULL,
    topic TEXT NOT NULL,
    payload TEXT NOT NULL,
    dev TEXT,
    seq INTEGER)""")                                  # 第0章の init_db と同じ表（コメントは省いた）
T0 = 1793000000.25                                # 2026 年 10 月ごろの UNIX 秒（例）
rng = np.random.default_rng(0)
rows = [(T0 + 5 * k, "lab/pico1/telemetry", json.dumps({"dev": "pico1", "seq": k, "temp_c": round(22 + rng.normal(0, 0.2), 2)}), "pico1", k)
        for k in range(720)]
rows += [(T0 + 4000 + 0.05 * k, "lab/bench/telemetry", json.dumps({"dev": "bench", "seq": k}), "bench", k) for k in range(50)]
rows += [(T0 + 4100, "lab/raw", "hello", None, None), (T0 + 4101, "lab/num", "[1, 2]", None, None)]
con.executemany("INSERT INTO messages (recv_unix, topic, payload, dev, seq) VALUES (?, ?, ?, ?, ?)", rows)
con.commit(); con.close()
with psycopg.connect() as c:
    n_before = c.execute("SELECT count(*) FROM messages").fetchone()[0]
print("SQLite に", len(rows), "件、PostgreSQL に今", n_before, "件")
rc1, out1 = run("sqlite2pg.py lab/lab.db --dry-run")

In [ ]:
# 確かめ（手順 13：--dry-run で件数を確かめる。bench は移さない）
ok("SQLite：722 件" in out1 and "同じ行：0 件" in out1 and "--dry-run なので書き込みません" in out1,
   "--dry-run：bench 以外の 722 件が対象で、PostgreSQL に同じ行はまだない。書き込まない")
with psycopg.connect() as c:
    ok(c.execute("SELECT count(*) FROM messages").fetchone()[0] == n_before, "--dry-run では件数が変わらない")
rc2, out2 = run("sqlite2pg.py lab/lab.db")
rc3, out3 = run("sqlite2pg.py lab/lab.db", check=False)       # うっかりもう一度
with psycopg.connect() as c:
    n_after = c.execute("SELECT count(*) FROM messages").fetchone()[0]
    first = c.execute("SELECT recv_unix, recv_time, payload FROM messages WHERE dev = 'pico1' AND seq = 0").fetchone()
    odd = c.execute("SELECT topic, payload FROM messages WHERE topic IN ('lab/raw', 'lab/num') AND recv_unix < %s ORDER BY topic",
                    (T0 + 5000,)).fetchall()      # 3 節で入れた同じトピックの行（いまの時刻）は除く
    nb = c.execute("SELECT count(*) FROM messages WHERE dev = 'bench'").fetchone()[0]
ok("722 件を移しました" in out2 and n_after == n_before + 722, f"本番：722 件を移した（{n_before} → {n_after} 件）")
ok(rc3 != 0 and "二重に入れないように止めました" in out3 and n_after == n_before + 722, "もう一度実行すると、最初と最後の記録が同じなので止まり、二重に入らない")
ok(nb == 0, "dev = bench の記録は移していない")
ok(abs(first[1].timestamp() - T0) < 1e-3 and first[0] == T0, "recv_time は to_timestamp(recv_unix) で、元の受け取った時刻（実行した時刻ではない）")
ok(odd == [("lab/num", {"value": [1, 2]}), ("lab/raw", {"raw": "hello"})], "JSON でない記録・dict でない JSON も、mqtt2pg.py と同じ形で移る")

## 6. バックアップと、復元できるかの確かめ（手順 14）

本文の `pi_docker.sh` の 7) と同じことを、同じ計算機の中で行います。`pg_dump -Fc`（PostgreSQL 独自の圧縮した形）でバックアップを取り、**別の DB（`labtest`）に復元して、件数と中身が同じか**を確かめます。本文では復元先を使い捨てのコンテナ（`pgtest`）にしましたが、ここでは同じサーバーの別の DB にします。

In [ ]:
os.makedirs("backup", exist_ok=True)
dump = f"backup/lab_{time.strftime('%Y-%m-%d')}.dump"
pg = lambda *a: subprocess.run([f"{PGBIN}/{a[0]}", "-h", "127.0.0.1", "-p", str(PGPORT), "-U", "lab", *a[1:]],
                               capture_output=True, text=True, check=True)
pg("pg_dump", "-d", "lab", "-Fc", "-f", dump)
print(dump, os.path.getsize(dump), "バイト")
with psycopg.connect(dbname="postgres", autocommit=True) as c:
    c.execute("DROP DATABASE IF EXISTS labtest")
pg("createdb", "labtest")
pg("pg_restore", "-d", "labtest", "--no-owner", dump)
q = "SELECT count(*), md5(string_agg(id::text || topic || payload::text, ',' ORDER BY id)) FROM messages"
with psycopg.connect() as c:
    a = c.execute(q).fetchone()
with psycopg.connect(dbname="labtest") as c:
    b = c.execute(q).fetchone()
print("lab    :", a)
print("labtest:", b)
ok(a == b and a[0] > 0, f"復元した DB の件数（{b[0]} 件）と中身（md5）が元と同じ")

## 7. 新旧の構成の遅れを比べる（手順 10、`bench_analysis.py`）

本文の `bench_e2e.py` は、旧構成（SQLite）と新構成（Docker＋PostgreSQL）に 100 通ずつのブロックを 10 回ずつ、**順番を乱数で混ぜて**送り、送った時刻から DB に入るまでの遅れを `bench.csv` に書きます。ここでは代わりに、`synth_bench.py` が同じ形の合成データを作ります。真の差は docker − sqlite = **+0.8 ms** で、順番が進むと遅れが上がって頭打ちになる**ドリフト**（最大 0.8 ms。温まる・たまるなどのまね）を入れました。実行は2回（`--runs 2`）です。

In [ ]:
run("synth_bench.py --out bench_synth.csv --runs 2 --blocks 10 --n 100 --seed 3")
rc, out = run("bench_analysis.py bench_synth.csv")
display(Image(filename="bench.png", width=900))

In [ ]:
# 確かめ（II-3-7：中央値は数 ms、まれに数十 ms。乱数で混ぜてあるので、ドリフトがあっても差の推定はかたよらない）
est = num(r"docker − sqlite = ([+-][\d.]+) ms", out)
lo, hi = num(r"95%区間 ([+-][\d.]+)〜", out), num(r"95%区間 [+-][\d.]+〜([+-][\d.]+)）", out)
d = pd.read_csv("bench_synth.csv")
ok(d["run"].nunique() == 2 and len(d) == 4000, "2 回の実行 × 20 ブロック × 100 通")
med = d.groupby("stack")["latency_ms"].median()
ok((1 < med).all() and (med < 10).all() and d["latency_ms"].max() > 20, f"中央値は数 ms（{med.round(2).to_dict()}）、まれに数十 ms（最大 {d['latency_ms'].max():.0f} ms）")
ok(lo < 0.8 < hi, f"docker − sqlite = {est:+.2f} ms、95% 区間 {lo:+.2f}〜{hi:+.2f} が真の差 +0.8 ms を含む")
ok("時間とともに変わる傾向がある" in out, "順番の係数 c で、ドリフトに気づける")

最後に、`bench_analysis.py` の説明にある【誤りの例】「片方の構成を先にまとめて測っていたら」を、答えのわかったデータで確かめます。`synth_bench.py --sequential` は、各実行で sqlite の 10 ブロックを先に、docker の 10 ブロックをあとに測ります。2 つの計画で、それぞれ合成データを 200 回作り直し、`bench_analysis.py` と同じ回帰（`med ~ 構成 + C(run) + order`）と、ブロックの平均の単純な差を計算します（約5秒）。

In [ ]:
import statsmodels.formula.api as smf
import synth_bench as sb


def estimate(sequential, seed):
    rng = np.random.default_rng(seed)
    rows = []
    for r in range(2):
        rows += sb.one_run(rng, sb.make_plan(10, seed * 100 + r, sequential), 100)
    d = pd.DataFrame(rows, columns=["run", "stack", "order", "seq", "latency_ms"])
    d["latency_ms"] = pd.to_numeric(d["latency_ms"])
    b = d.dropna().groupby(["run", "stack", "order"])["latency_ms"].median().rename("med").reset_index()
    m = smf.ols("med ~ C(stack, Treatment('sqlite')) + C(run) + order", b).fit()     # bench_analysis.py と同じ式
    name = [p for p in m.params.index if p.startswith("C(stack")][0]
    lo, hi = m.conf_int().loc[name]
    nv = b.groupby("stack")["med"].mean()
    return m.params[name], lo <= sb.DIFF <= hi, nv["docker"] - nv["sqlite"], hi - lo


N_SIM = 200            # 増やすと時間がかかる（200 回で約5秒）
res = {}
for seq in [False, True]:
    r = np.array([estimate(seq, s) for s in range(100, 100 + N_SIM)])
    res[seq] = dict(reg=r[:, 0].mean(), cover=r[:, 1].mean(), naive=r[:, 2].mean(), sd=r[:, 0].std(), width=r[:, 3].mean())
    print(f"{'まとめて順に（誤り）' if seq else '乱数で混ぜる（本文）'}：回帰の推定の平均 {res[seq]['reg']:+.3f} ms、"
          f"95% 区間が真の値を含んだ割合 {res[seq]['cover']:.0%}、単純な差の平均 {res[seq]['naive']:+.3f} ms（真の値 +{sb.DIFF} ms）、"
          f"回帰の推定のばらつき（SD）{res[seq]['sd']:.3f} ms、95% 区間の幅の平均 {res[seq]['width']:.3f} ms")

In [ ]:
ok(abs(res[False]["reg"] - 0.8) < 0.03 and abs(res[False]["naive"] - 0.8) < 0.03 and res[False]["cover"] > 0.9,
   "乱数で混ぜると、回帰でも単純な差でも、平均して真の差 +0.8 ms が出る（かたよらない）")
ok(res[True]["naive"] - 0.8 > 0.2, f"まとめて測ると、単純な差は {res[True]['naive'] - 0.8:+.2f} ms かたよる（ドリフトを構成の差と取り違える）")
ok(res[True]["width"] > 1.5 * res[False]["width"],
   f"まとめて測った計画では、順番の項を入れた回帰の区間の幅が {res[True]['width'] / res[False]['width']:.1f} 倍に広がる（ドリフトと構成の差が重なり、精度が落ちる）")
print(f"回帰のかたより：乱数で混ぜる {res[False]['reg'] - 0.8:+.3f} ms、まとめて順に {res[True]['reg'] - 0.8:+.3f} ms（小さいが残る）")

- 乱数で混ぜた計画では、ドリフトがあっても、構成の差の推定はかたよりません（ランダム化の効果）。順番の項 `order` は、ドリフトがあることに気づくための道具です。
- まとめて測った計画では、ドリフトと構成の差がほとんど重なります。順番の項を入れた回帰では、かたよりは小さくなりますが少し残り、区間の幅はおよそ2倍に広がります（同じ回数測っても、分かることが少なくなる）。**あとの解析で直すより、測る順番を乱数で決めるほうが確実**です。

In [ ]:
# 片付け：PostgreSQL を止める（もう一度試すときは、1 節から実行し直す）
r = subprocess.run(as_pg([f"{PGBIN}/pg_ctl", "-D", DATA, "-w", "-m", "fast", "stop"]), capture_output=True, text=True)
print(r.stdout)
ok(r.returncode == 0, "PostgreSQL を止めた")

## おわりに

- 受信 API と MQTT の記録係は、どちらも受け取った JSON を `payload`（JSONB）にそのまま入れ、`dev`・`seq` を列にも入れます。JSON でない文字も捨てずに保存するので、あとで何が届いていたかを調べられます。
- 移行のプログラムは、`--dry-run` で件数を確かめてから実行し、二重に入れないように止まります。受け取った時刻（`recv_unix`）は元のまま移すので、Grafana の時間の範囲で過去の記録も見られます。
- バックアップは、別の DB に復元して件数と中身を確かめて、初めてバックアップです。
- 新旧の比較は、ブロックを独立な単位にし、測る順番を乱数で混ぜます。
- このノートブックに入れていないもの：Docker と compose、ufw を素通りすることの確かめ、Mosquitto、Grafana の画面、Pi の上での実測と移行。これらは教科書の手順のとおりに Pi で行います。